# Water and groundwater regional sensitivity analysis (target 2020)

This notebook is separate from `cropland_soil_group_wetland_excluded_oof_shap_target2020.ipynb`.
It does not modify the basic-model notebook.

## Scope

- Reuse the saved basic-model Spatial OOF predictions.
- Compare Fan et al. water-table depth, WaterGAP groundwater recharge, GloFAS river thresholds, and surrounding flow.
- Exclude FAO GMIA and every observed irrigation-area variable from model inputs. Irrigated/equipped area is a subset of cropland and therefore encodes cropland presence even when divided by total grid-cell area; dividing by cropland area creates an even more direct circularity.
- Retain the GAEZ *potential* irrigation calorie gain already in the basic model. It is a counterfactual agro-ecological potential derived without observed cropland extent, not an observed irrigation-area measure.
- Compare every retained candidate against the same target-2020 sample and Spatial OOF folds.


In [ ]:
from __future__ import annotations

import calendar
import re
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr


ROOT = Path("/work/tsuda")
GAEZ_DIR = ROOT / "GAEZ"
WATER_DIR = GAEZ_DIR / "WaterData"
GROUNDWATER_DIR = WATER_DIR / "groundwater"
WTD_RAW_DIR = GROUNDWATER_DIR / "fan_wtd" / "raw_continents"
WTD_SUBSET_DIR = GROUNDWATER_DIR / "fan_wtd" / "regional_subsets"
WATERGAP_RAW_DIR = GROUNDWATER_DIR / "watergap_2_2d" / "raw"
WATERGAP_PROCESSED_DIR = GROUNDWATER_DIR / "watergap_2_2d" / "processed"

BASIC_OUTPUT_DIR = (
    GAEZ_DIR
    / "CroplandRegression"
    / "soil_group_calorie_only_50km_target2020"
)
BASIC_OOF_PATH = (
    BASIC_OUTPUT_DIR
    / "soil_comparison_oof_predictions.csv.gz"
)
ANALYSIS_OUTPUT_DIR = (
    GAEZ_DIR
    / "CroplandRegression"
    / "water_irrigation_regional_sensitivity_target2020"
)

for directory in [
    WTD_SUBSET_DIR,
    WATERGAP_PROCESSED_DIR,
    ANALYSIS_OUTPUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

REGION_BOXES = {
    "usa_around_100w": (-110.0, -90.0, 25.0, 50.0),
    "nile_region": (-10.0, 40.0, 5.0, 32.0),
    "yellow_river_region": (100.0, 125.0, 25.0, 45.0),
    "japan_region": (122.0, 154.0, 20.0, 47.0),
}

REGION_CONTINENT = {
    "usa_around_100w": "north_america",
    "nile_region": "africa",
    "yellow_river_region": "eurasia",
    "japan_region": "eurasia",
}

WTD_RAW_PATHS = {
    "africa": WTD_RAW_DIR / "AFRICA_WTD_annualmean.nc",
    "eurasia": WTD_RAW_DIR / "EURASIA_WTD_annualmean.nc",
    "north_america": WTD_RAW_DIR / "NAMERICA_WTD_annualmean.nc",
    "oceania": WTD_RAW_DIR / "OCEANIA_WTD_annualmean.nc",
    "south_america": WTD_RAW_DIR / "SAMERICA_WTD_annualmean.nc",
}

WATERGAP_FILES = {
    "total": (
        WATERGAP_RAW_DIR
        / "watergap_22d_WFDEI-GPCC_histsoc_qr_monthly_1901_2016.nc4"
    ),
    "diffuse": (
        WATERGAP_RAW_DIR
        / "watergap_22d_WFDEI-GPCC_histsoc_qrdif_monthly_1901_2016.nc4"
    ),
    "surface_water_body": (
        WATERGAP_RAW_DIR
        / "watergap_22d_WFDEI-GPCC_histsoc_qrswb_monthly_1901_2016.nc4"
    ),
}

missing_files = [
    path
    for path in [
        *WTD_RAW_PATHS.values(),
        *WATERGAP_FILES.values(),
        BASIC_OOF_PATH,
    ]
    if not path.exists()
]
if missing_files:
    raise FileNotFoundError(
        "Required input files are missing:\n"
        + "\n".join(str(path) for path in missing_files)
    )

print("All downloaded groundwater inputs are present.")
for path in [*WTD_RAW_PATHS.values(), *WATERGAP_FILES.values()]:
    print(f"{path.name}: {path.stat().st_size / 1024**2:.1f} MiB")
print("Basic OOF:", BASIC_OOF_PATH)


## 1. Fan water-table depth

The source stores water-table elevation relative to the land surface as non-positive metres.
The derived variable below is positive depth below the land surface in metres.
Regional subsets retain the native 30-arc-second resolution.


In [ ]:
def coordinate_slice(coordinate, lower, upper):
    values = np.asarray(coordinate.values)
    if values[0] <= values[-1]:
        return slice(lower, upper)
    return slice(upper, lower)


wtd_manifest_rows = []
for continent, raw_path in WTD_RAW_PATHS.items():
    wtd_manifest_rows.append({
        "continent": continent,
        "local_path": str(raw_path),
        "size_bytes": raw_path.stat().st_size,
        "source": "Fan global water table depth",
        "definition": "modeled static water-table depth",
    })

wtd_manifest = pd.DataFrame(wtd_manifest_rows)
wtd_manifest.to_csv(
    GROUNDWATER_DIR / "fan_wtd_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

wtd_summary_rows = []

for region_name, bbox in REGION_BOXES.items():
    west, east, south, north = bbox
    continent = REGION_CONTINENT[region_name]
    raw_path = WTD_RAW_PATHS[continent]

    with xr.open_dataset(raw_path, decode_times=False) as ds:
        source = ds["WTD"]
        if "time" in source.dims:
            source = source.isel(time=0, drop=True)

        lat_selection = coordinate_slice(
            ds["lat"], south, north
        )
        lon_selection = coordinate_slice(
            ds["lon"], west, east
        )
        subset = source.sel(
            lat=lat_selection,
            lon=lon_selection,
        ).astype(np.float32)
        domain_mask = ds["mask"].sel(
            lat=lat_selection,
            lon=lon_selection,
        )

        # Fan WTD uses zero at the land surface and negative values below it.
        depth = (-subset).where(
            np.isfinite(subset) & (domain_mask == 1)
        )
        depth = depth.where(depth >= 0)
        depth = depth.rename("water_table_depth_m")
        depth.attrs = {
            "long_name": "water-table depth below land surface",
            "units": "m",
            "source": "Fan global water table depth",
            "source_file": raw_path.name,
            "region": region_name,
            "sign_conversion": "water_table_depth_m = -WTD",
        }
        depth = depth.load()

    output_path = WTD_SUBSET_DIR / f"fan_wtd_{region_name}.nc"
    depth.to_dataset().to_netcdf(
        output_path,
        encoding={
            "water_table_depth_m": {
                "zlib": True,
                "complevel": 4,
                "dtype": "float32",
            }
        },
    )

    values = depth.values
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        raise RuntimeError(f"No valid WTD values in {region_name}")

    wtd_summary_rows.append({
        "region": region_name,
        "continent": continent,
        "path": str(output_path),
        "n_finite": int(finite.size),
        "min_m": float(np.min(finite)),
        "median_m": float(np.median(finite)),
        "p90_m": float(np.percentile(finite, 90)),
        "max_m": float(np.max(finite)),
    })
    print("Saved:", output_path)

wtd_region_summary = pd.DataFrame(wtd_summary_rows)
wtd_region_summary.to_csv(
    GROUNDWATER_DIR / "fan_wtd_regional_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
display(wtd_region_summary.round(3))


## 2. WaterGAP groundwater recharge

WaterGAP uses a non-CF-compliant `months since 1901-01-01` time axis.
It is opened with `decode_times=False`, converted to calendar year/month explicitly,
and averaged over 2000–2010 using the number of days in each month as weights.

The source flux unit `kg m-2 s-1` is equivalent to mm s-1 of water.
The outputs are annualized to mm yr-1.


In [ ]:
RECHARGE_START_YEAR = 2000
RECHARGE_END_YEAR = 2010
SECONDS_PER_YEAR = 365.2425 * 24 * 60 * 60

EXPECTED_VARIABLES = {
    "total": "qr",
    "diffuse": "qrdif",
    "surface_water_body": "qrswb",
}


def month_year_arrays(time_coordinate):
    units = str(time_coordinate.attrs.get("units", ""))
    match = re.fullmatch(
        r"months since\s+(\d{4})-(\d{2})-(\d{2})",
        units.strip(),
    )
    if match is None:
        raise ValueError(f"Unsupported WaterGAP time units: {units}")

    origin_year = int(match.group(1))
    origin_month = int(match.group(2))
    month_offsets = np.rint(
        np.asarray(time_coordinate.values, dtype=float)
    ).astype(int)
    absolute_month = (origin_month - 1) + month_offsets
    years = origin_year + absolute_month // 12
    months = absolute_month % 12 + 1
    return years, months


recharge_summary_rows = []
recharge_manifest_rows = []

for kind, raw_path in WATERGAP_FILES.items():
    variable_name = EXPECTED_VARIABLES[kind]

    with xr.open_dataset(raw_path, decode_times=False) as ds:
        if variable_name not in ds.data_vars:
            raise KeyError(
                f"{variable_name} is absent from {raw_path.name}: "
                f"{list(ds.data_vars)}"
            )

        data = ds[variable_name]
        years, months = month_year_arrays(ds["time"])
        selected_indices = np.flatnonzero(
            (years >= RECHARGE_START_YEAR)
            & (years <= RECHARGE_END_YEAR)
        )
        if selected_indices.size == 0:
            raise RuntimeError(
                f"No WaterGAP months in {RECHARGE_START_YEAR}-"
                f"{RECHARGE_END_YEAR}"
            )

        selected = data.isel(time=selected_indices)
        selected_years = years[selected_indices]
        selected_months = months[selected_indices]
        day_weights = np.array(
            [
                calendar.monthrange(int(year), int(month))[1]
                for year, month in zip(
                    selected_years,
                    selected_months,
                    strict=True,
                )
            ],
            dtype=np.float64,
        )
        weights = xr.DataArray(
            day_weights,
            dims=["time"],
            coords={"time": selected["time"]},
        )

        mean_rate = selected.weighted(weights).mean(
            dim="time",
            skipna=True,
        )
        recharge = (
            mean_rate * SECONDS_PER_YEAR
        ).astype(np.float32)

        if float(recharge["lon"].max()) > 180:
            normalized_lon = (
                (recharge["lon"] + 180.0) % 360.0
            ) - 180.0
            recharge = (
                recharge.assign_coords(lon=normalized_lon)
                .sortby("lon")
            )

        output_variable = (
            f"groundwater_recharge_{kind}_mm_yr"
        )
        recharge = recharge.rename(output_variable)
        recharge.attrs = {
            "long_name": (
                f"{kind.replace('_', ' ')} groundwater recharge"
            ),
            "units": "mm yr-1",
            "source": "WaterGAP 2.2d Standard model output",
            "source_doi": "10.1594/PANGAEA.918447",
            "source_variable": variable_name,
            "source_units": str(data.attrs.get("units", "")),
            "mean_period": (
                f"{RECHARGE_START_YEAR}-{RECHARGE_END_YEAR}"
            ),
            "temporal_weighting": "calendar days per month",
        }
        recharge = recharge.load()

    output_path = (
        WATERGAP_PROCESSED_DIR
        / (
            f"watergap_{kind}_recharge_mm_yr_"
            f"{RECHARGE_START_YEAR}_{RECHARGE_END_YEAR}.nc"
        )
    )
    recharge.to_dataset().to_netcdf(
        output_path,
        encoding={
            output_variable: {
                "zlib": True,
                "complevel": 4,
                "dtype": "float32",
            }
        },
    )

    values = recharge.values
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        raise RuntimeError(f"No valid recharge values for {kind}")

    recharge_summary_rows.append({
        "kind": kind,
        "source_variable": variable_name,
        "path": str(output_path),
        "unit": "mm yr-1",
        "mean_period": (
            f"{RECHARGE_START_YEAR}-{RECHARGE_END_YEAR}"
        ),
        "n_finite": int(finite.size),
        "min": float(np.min(finite)),
        "median": float(np.median(finite)),
        "p90": float(np.percentile(finite, 90)),
        "max": float(np.max(finite)),
    })
    recharge_manifest_rows.append({
        "kind": kind,
        "filename": raw_path.name,
        "local_path": str(raw_path),
        "size_bytes": raw_path.stat().st_size,
        "source": "WaterGAP 2.2d",
        "source_doi": "10.1594/PANGAEA.918447",
    })
    print("Saved:", output_path)

recharge_summary = pd.DataFrame(recharge_summary_rows)
recharge_summary.to_csv(
    GROUNDWATER_DIR / "watergap_recharge_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

recharge_manifest = pd.DataFrame(recharge_manifest_rows)
recharge_manifest.to_csv(
    GROUNDWATER_DIR
    / "watergap_selected_recharge_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

groundwater_manifest = pd.concat(
    [
        wtd_manifest.assign(dataset_group="Fan_WTD"),
        recharge_manifest.assign(
            dataset_group="WaterGAP_recharge"
        ),
    ],
    ignore_index=True,
    sort=False,
)
groundwater_manifest.to_csv(
    GROUNDWATER_DIR / "groundwater_download_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

display(recharge_summary.round(3))


## 3. Create a harmonized basic-model regional OOF summary

This uses only the already-saved target-2020 OOF prediction table and the same bounding boxes as the groundwater subsets.
It does not retrain or change the basic model. Earlier regional figures used different boxes, so their cell counts need not match this table.


In [ ]:
basic_oof = pd.read_csv(
    BASIC_OOF_PATH,
    compression="gzip",
)

prediction_column = "pred_soil_group_wetland_excluded"
required_oof_columns = {
    "lat",
    "lon",
    "cropland_fraction",
    prediction_column,
}
missing_oof_columns = (
    required_oof_columns - set(basic_oof.columns)
)
if missing_oof_columns:
    raise KeyError(
        f"Missing basic OOF columns: {sorted(missing_oof_columns)}"
    )


def regional_regression_summary(frame, prediction_col):
    observed = frame["cropland_fraction"].to_numpy(float)
    predicted = frame[prediction_col].to_numpy(float)
    valid = np.isfinite(observed) & np.isfinite(predicted)
    observed = observed[valid]
    predicted = predicted[valid]
    residual = predicted - observed
    observed_mean = float(np.mean(observed))
    denominator = float(
        np.sum((observed - observed_mean) ** 2)
    )
    r2 = (
        1.0
        - float(np.sum((observed - predicted) ** 2))
        / denominator
        if denominator > 0
        else np.nan
    )
    return {
        "n_cells": int(len(observed)),
        "observed_mean": observed_mean,
        "predicted_mean": float(np.mean(predicted)),
        "bias_predicted_minus_observed": float(
            np.mean(residual)
        ),
        "rmse": float(np.sqrt(np.mean(residual**2))),
        "mae": float(np.mean(np.abs(residual))),
        "r2": r2,
        "overprediction_share_pct": float(
            100 * np.mean(residual > 0)
        ),
        "underprediction_share_pct": float(
            100 * np.mean(residual < 0)
        ),
        "absolute_error_p95": float(
            np.percentile(np.abs(residual), 95)
        ),
    }


regional_rows = []
for region_name, bbox in REGION_BOXES.items():
    west, east, south, north = bbox
    region_frame = basic_oof.loc[
        basic_oof["lon"].between(west, east)
        & basic_oof["lat"].between(south, north)
    ].copy()
    if region_frame.empty:
        raise RuntimeError(
            f"No basic OOF cells in {region_name}"
        )
    regional_rows.append({
        "region": region_name,
        **regional_regression_summary(
            region_frame,
            prediction_column,
        ),
    })

regional_oof_summary = pd.DataFrame(regional_rows)
regional_oof_summary.to_csv(
    ANALYSIS_OUTPUT_DIR
    / "basic_regional_oof_error_summary_groundwater_boxes_2020.csv",
    index=False,
    encoding="utf-8-sig",
)
display(regional_oof_summary.round(4))


## 4. Sensitivity-analysis scope after excluding observed irrigation area

The comparison is implemented outside the basic notebook and uses the saved target-2020 sample and Spatial OOF folds. The retained candidates are:

1. Replace the basic 10 m³/s river-distance threshold with 1 or 0.1 m³/s.
2. Add the latitude-aware 50-km mean of `log1p(GloFAS p10 flow)`.
3. Add Fan et al. groundwater-table depth.
4. Add WaterGAP groundwater recharge.
5. Add groundwater-table depth and recharge together.
6. Fit a no-irrigation combined candidate using only changes that passed the predeclared regional-selection rule.

FAO GMIA is not used in any candidate or combined model. HydroRIVERS is not mixed into this table because its mean-discharge definition differs from GloFAS p10 reliable discharge; it remains a separate sensitivity definition.


## 5. Run or resume the Spatial OOF comparison

This cell is independent of the cells above. Completed model predictions are reused from `oof_checkpoints`, so rerunning it does not retrain finished models. The basic model notebook is not modified.


In [ ]:
from pathlib import Path
import runpy

SENSITIVITY_SCRIPT = Path(
    "/work/tsuda/GAEZ/run_water_irrigation_regional_sensitivity_target2020.py"
)
if not SENSITIVITY_SCRIPT.exists():
    raise FileNotFoundError(SENSITIVITY_SCRIPT)

runpy.run_path(str(SENSITIVITY_SCRIPT), run_name="__main__")


## 6. Read the completed regional comparison

Positive `rmse_improvement_vs_baseline` and `r2_gain_vs_baseline` mean improvement. `clear_improvement` means the 95% spatial-block bootstrap interval for candidate RMSE minus baseline RMSE is entirely below zero.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

RESULT_DIR = Path(
    "/work/tsuda/GAEZ/CroplandRegression/"
    "water_irrigation_regional_sensitivity_target2020"
)
regional_comparison = pd.read_csv(
    RESULT_DIR / "regional_model_error_comparison_2020.csv"
)
best_individual_by_region = pd.read_csv(
    RESULT_DIR / "best_individual_change_by_region_2020.csv"
)

display(
    best_individual_by_region[[
        "region_label",
        "model_label",
        "rmse_improvement_vs_baseline",
        "r2_gain_vs_baseline",
        "improvement_class",
    ]].round(6)
)
display(Image(filename=str(
    RESULT_DIR / "regional_error_change_by_model_2020.png"
)))


## Final global water-specification decision

Fan water-table depth and WaterGAP recharge are fixed in every candidate. The remaining factorial comparison crosses river thresholds of 10, 1, and 0.1 m³/s with inclusion/exclusion of the surrounding 50-km GloFAS p10-flow feature.

- The minimum point-estimate global RMSE is obtained by **10 m³/s + 50-km GloFAS** (`RMSE = 0.133559`).
- However, adding the 50-km feature to the 10 m³/s model changes RMSE by only `-0.0000239`; the paired spatial-block bootstrap 95% interval is `[-0.000401, +0.000356]`, with only `55.7%` probability of lower RMSE. This is not supported improvement.
- The 10 and 1 m³/s thresholds are effectively tied (`RMSE difference ≈ 0.0000006`), and 0.1 m³/s provides no supported gain.
- Therefore the parsimonious global recommendation is: **retain the 10 m³/s river threshold and do not add the 50-km GloFAS feature**, while always adding groundwater-table depth and groundwater recharge.
- This recommended water specification improves global RMSE from `0.134779` to `0.133583` and global R² by `+0.006550`. Its global improvement over the basic model is supported by the paired spatial-block bootstrap (`95% CI for ΔRMSE = [-0.002119, -0.000315]`).

Output files:

- `global_water_factorial_comparison_2020.csv`
- `global_water_specification_block_bootstrap_2020.csv`
- `global_water_specification_selection.json`
- `global_selected_water_model_regional_performance_2020.csv`
- `regional_error_change_maps/*_global_selected_water_specification_error_change_grid.png`

All files are under `/work/tsuda/GAEZ/CroplandRegression/water_irrigation_regional_sensitivity_target2020/`.
